# Customizing priors and combining the Thiele-Innes and standard parameterizations

> NOTE: This tutorial assumes that you have read through the {doc}`getting started tutorial <0-getting-started>` and are familiar with the basic workflow.

This tutorial demonstrates a two-stage workflow for fitting Gaia epoch astrometry:

1. **Rejection sampling with the Thiele-Innes parameterization.** The Thiele-Innes parameterization replaces the four Campbell orientation parameters $(\omega, \Omega, \cos i, a_0)$ with four Thiele-Innes constants $(A, B, F, G)$ that enter the along-scan model **linearly**. This reduces the nonlinear parameter space from 6-D to 3-D (period, eccentricity, phase), making rejection sampling much more efficient.
2. **Refining with MCMC in the standard parameterization.** Once rejection sampling has localized the posterior, we convert the accepted samples back to physical Campbell elements and continue with MCMC using the {py:class}`~harv.models.parameterizations.gaia.StandardGaiaAstrometry` parameterization, which samples the physical orbital elements directly.

Along the way we also show how to build a fully custom {py:class}`~harv.HarvPrior`.

In [ ]:
import astropy.table as at
import jax
import numpyro.distributions as dist
import quaxed.numpy as jnp
from unxt import Q

import harv
from harv.models.parameterizations.gaia import (
    StandardGaiaAstrometry,
    ThieleInnesGaiaAstrometry,
)
from harv.models.priors.custom_priors import (
    ParallaxDependentProperMotionPrior,
    PeriodDependentSemiMajorAxisPrior,
)

jax.config.update("jax_enable_x64", True)

%matplotlib inline

In [ ]:
truths = at.Table.read("../data/gaia-DACE-ohp/injected_solutions.csv")
truth_row = truths[4]

In [ ]:
truth_row["comment"]

In [ ]:
tbl = at.Table.read(f"../data/gaia-DACE-ohp/{truth_row['filename']}")

fdtype = jnp.float64
data = harv.GaiaAstrometryData(
    time=Q(tbl["obs_time_tcb"].astype(fdtype), "day"),
    al_position=Q(tbl["centroid_pos_al"].astype(fdtype), "mas"),
    al_position_err=Q(tbl["centroid_pos_error_al"].astype(fdtype), "mas"),
    scan_angle=Q(tbl["scan_pos_angle"].astype(fdtype), "rad"),
    parallax_factor=tbl["parallax_factor_al"].astype(fdtype),
)
data

Let's start by plotting the data:

In [ ]:
_ = data.plot(relative_to_t_ref=True)

This is a single Gaia source with 76 along-scan epoch measurements. Each point is a one-dimensional position measured along the satellite's scan direction at a particular scan angle. The orbital signal is the small periodic wobble superimposed on the linear proper-motion drift and the parallax ellipse.

## Setting up the Thiele-Innes parameterization

The standard Gaia astrometry parameterization (`StandardGaiaAstrometry`, used in the getting-started tutorial) has **six nonlinear parameters**: `period`, `eccentricity`, `phase_peri`, `arg_peri`, `lon_asc_node`, and `cos_i`. The four orientation parameters `arg_peri`, `lon_asc_node`, `cos_i`, and `semi_major_axis` enter the along-scan model through the Thiele-Innes constants $A, B, F, G$. We can exploit this by switching to {py:class}`~harv.models.parameterizations.gaia.ThieleInnesGaiaAstrometry`, which promotes $A, B, F, G$ to **linear** parameters, leaving only three nonlinear parameters.

First, create the parameterization from the data. The `from_data` class method automatically sets an appropriate numerical floor `a_floor` based on the data uncertainties:


In [ ]:
# Create the parameterization; a_floor is set from the data noise level
# ti_param = ThieleInnesGaiaAstrometry.from_data(data, apply_jacobian_correction=False)
ti_param = ThieleInnesGaiaAstrometry()
ti_param

Now build a {py:class}`~harv.HarvPrior` for the Thiele-Innes parameter set. The nonlinear prior only needs three parameters. For each Thiele-Innes constant we use the same period- and parallax-scaled Gaussian as the `semi_major_axis` prior in the standard parameterization, because the TI constants have the same angular units and the same physical interpretation up to orientation factors of order unity.

Note that `parallax` is kept as an **explicitly sampled** linear parameter (by omitting it from `marginalized_names` in the sampler below). This is required so that the `PeriodDependentSemiMajorAxisPrior` and `ParallaxDependentProperMotionPrior` callables can look up the current parallax value when computing their conditional scales. Update the `parallax` Normal parameters to match the catalog values for your target.

In [ ]:
# Shared prior for each TI constant: period-and-parallax-scaled Gaussian
ti_prior = PeriodDependentSemiMajorAxisPrior(sigma_a0=Q(1, "AU"), P0=Q(300, "day"))

prior = harv.HarvPrior(
    nonlinear_priors={
        "period": harv.QD(dist.LogUniform(10.0, 1e3), "day"),
        "eccentricity": dist.Beta(0.867, 3.03),  # Kipping (2013)
        "phase_peri": dist.Uniform(0.0, 1.0),
    },
    linear_prior={
        "ra0": harv.QD(dist.Normal(0.0, 100.0), "mas"),
        "dec0": harv.QD(dist.Normal(0.0, 100.0), "mas"),
        "pmra": ParallaxDependentProperMotionPrior(sigma_v0=Q(100.0, "km/s")),
        "pmdec": ParallaxDependentProperMotionPrior(sigma_v0=Q(100.0, "km/s")),
        "parallax": harv.QD(dist.Normal(44.3, 0.05), "mas"),  # update for your target
        "ti_A": ti_prior,
        "ti_B": ti_prior,
        "ti_F": ti_prior,
        "ti_G": ti_prior,
    },
)

Let's confirm the breakdown of which parameters are nonlinear vs. linear with the Thiele-Innes prior:

In [ ]:
print(prior.nonlinear_priors.keys())
print(prior.linear_prior.keys())

With the Thiele-Innes parameterization the prior has only **three** nonlinear parameters (`period`, `eccentricity`, `phase_peri`) instead of the usual six. The four Thiele-Innes constants (`ti_A`, `ti_B`, `ti_F`, `ti_G`) are now linear parameters alongside the standard astrometric parameters. Parallax is kept as an explicitly sampled (non-marginalized) linear parameter so that the parallax-scaled proper-motion and semi-major-axis priors can condition on it.

## Running the rejection sampler

With data and prior now specified, we can create a {py:class}`harv.RejectionSampler` to manage running the rejection sampling:

In [ ]:
model = harv.GaiaAstrometryModel(parameterization=ti_param)

sampler = harv.RejectionSampler(
    prior,
    model,
    # parallax is intentionally excluded: it is sampled explicitly so that the
    # parallax-dependent priors on pmra, pmdec, and the TI constants can use it
    marginalized_names=("ra0", "dec0", "pmra", "pmdec", "ti_A", "ti_B", "ti_F", "ti_G"),
)

And then we can call `.run()` to run the sampler. The default specification for the sampler requires the number of prior samples to draw (`n_prior_samples`) and the maximum number of posterior samples to return (`max_posterior_samples`). You can also set a random number seed (`seed`) for reproducibility.

In [ ]:
samples = sampler.run(
    data,
    n_prior_samples=1_000_000,
    max_posterior_samples=1024,
    seed=123,
)
samples

The Thiele-Innes parameterization makes rejection sampling efficient: because only three parameters are nonlinear, the acceptance rate for a given number of prior draws is much higher than with the six-dimensional standard parameterization.

Let's inspect the posterior over the nonlinear and explicitly-sampled parameters with {py:meth}`~harv.samplers.Samples.plot_corner`:

In [ ]:
axes = samples.plot_corner(
    ["period", "eccentricity", "parallax", "pmra", "pmdec"],
    visuals={"scatter": {"alpha": 0.75, "s": 8}},
)

In [ ]:
axes = samples.plot_corner(
    ["ti_A", "ti_B", "ti_F", "ti_G", "phase_peri"],
    visuals={"scatter": {"alpha": 0.75, "s": 8}},
)

## Converting Thiele-Innes constants to Campbell elements

The Thiele-Innes constants $(A, B, F, G)$ are convenient for sampling, but the physical orbital elements -- semi-major axis $a_0$, argument of periastron $\omega$, longitude of the ascending node $\Omega$, and inclination $i$ -- are usually what we want to report and to sample with MCMC.

{py:meth}`~harv.samplers.Samples.thiele_innes_to_campbell` converts a Thiele-Innes `Samples` object into the standard (Campbell) parameterization, replacing the linear parameters `ti_A, ti_B, ti_F, ti_G` with `semi_major_axis` (linear) and `arg_peri, lon_asc_node, cos_i` (nonlinear):

In [ ]:
campbell_samples = samples.thiele_innes_to_campbell()

print("nonlinear:", list(campbell_samples.nonlinear.keys()))
print("linear:", list(campbell_samples.linear.keys()))
campbell_samples

The converted samples now carry the physical orientation parameters. Note that `thiele_innes_to_campbell` adopts the $\cos i \ge 0$ convention and wraps `arg_peri` and `lon_asc_node` into $[0, 2\pi)$. Here is the posterior in the physical elements:

In [ ]:
axes = campbell_samples.plot_corner(
    ["period", "eccentricity", "semi_major_axis", "cos_i", "lon_asc_node"],
    visuals={"scatter": {"alpha": 0.75, "s": 8}},
)

## Refining with MCMC in the standard parameterization

Rejection sampling efficiently localizes the posterior, but with sparse data the accepted samples can be few and lumpy. To obtain a smooth, well-sampled posterior we continue with MCMC, *warm-started* from the rejection-sampling output.

For MCMC we use the **standard** parameterization, which samples the physical Campbell elements directly (no Thiele-Innes Jacobian correction is needed). We first build a {py:class}`~harv.HarvPrior` for the standard parameter set. The `period`, `eccentricity`, `phase_peri`, and astrometric linear priors are identical to the Thiele-Innes prior above; we add isotropic-orbit priors for the three orientation parameters (`arg_peri`, `lon_asc_node`, `cos_i`) and reuse the period-scaled prior for `semi_major_axis`.

In [ ]:
standard_prior = harv.HarvPrior(
    nonlinear_priors={
        "period": harv.QD(dist.LogUniform(10.0, 1e3), "day"),
        "eccentricity": dist.Beta(0.867, 3.03),  # Kipping (2013)
        "phase_peri": dist.Uniform(0.0, 1.0),
        "arg_peri": harv.QD(dist.Uniform(0.0, 2.0 * jnp.pi), "rad"),
        "lon_asc_node": harv.QD(dist.Uniform(0.0, 2.0 * jnp.pi), "rad"),
        "cos_i": dist.Uniform(-1.0, 1.0),
    },
    linear_prior={
        "ra0": harv.QD(dist.Normal(0.0, 100.0), "mas"),
        "dec0": harv.QD(dist.Normal(0.0, 100.0), "mas"),
        "pmra": ParallaxDependentProperMotionPrior(sigma_v0=Q(100.0, "km/s")),
        "pmdec": ParallaxDependentProperMotionPrior(sigma_v0=Q(100.0, "km/s")),
        "parallax": harv.QD(dist.Normal(44.3, 0.05), "mas"),  # update for your target
        "semi_major_axis": PeriodDependentSemiMajorAxisPrior(
            sigma_a0=Q(1, "AU"), P0=Q(300, "day")
        ),
    },
)

Now create a {py:class}`~harv.NumpyroSampler` with the standard model and run MCMC, passing the converted Campbell samples as `init_samples` to set the starting position of each chain. We request `return_logprobs=True` so the output carries per-sample log-probabilities (used below).

In [ ]:
mcmc_model = harv.GaiaAstrometryModel(parameterization=StandardGaiaAstrometry())

mcmc_sampler = harv.NumpyroSampler(
    standard_prior,
    mcmc_model,
    # keep parallax explicitly sampled, as in the rejection step, so the
    # parallax-dependent priors can condition on it
    marginalized_names=("ra0", "dec0", "pmra", "pmdec", "semi_major_axis"),
)

mcmc_samples = mcmc_sampler.run(
    data,
    init_samples=campbell_samples,
    seed=42,
    num_warmup=500,
    num_samples=1000,
    num_chains=4,
    return_logprobs=True,
)
mcmc_samples

The MCMC posterior is smooth and densely sampled compared to the rejection-sampling output. Because the chains were warm-started inside the modes localized by rejection sampling, MCMC converges quickly:

In [ ]:
axes = mcmc_samples.plot_corner(
    ["period", "eccentricity", "semi_major_axis", "cos_i", "parallax"],
    visuals={"scatter": {"alpha": 0.5, "s": 6}},
)

## Assessing the fit

Because we ran the MCMC with `return_logprobs=True`, the returned `Samples` object stores per-sample `ln_likelihood` and `ln_prior`. This enables {py:meth}`~harv.samplers.Samples.map_sample` (the maximum a posteriori sample) and the `ln_posterior` property.

We can also compute a per-sample reduced $\chi^2$ with {py:meth}`~harv.samplers.Samples.reduced_chi2` -- a goodness-of-fit statistic distinct from the marginal log-likelihood. A value near 1 indicates the model fits the data within the quoted uncertainties.

In [ ]:
# The MAP (maximum a posteriori) sample -- available because we passed
# return_logprobs=True to run():
best = mcmc_samples.map_sample()
print("MAP period:", best["period"])

# Per-sample reduced chi-squared of the model against the data:
reduced_chi2 = mcmc_samples.reduced_chi2(data, mcmc_model)
print(f"reduced chi^2: median = {float(jnp.median(reduced_chi2)):.2f}")

## Next steps

This tutorial combined two samplers: rejection sampling in the efficient Thiele-Innes parameterization to localize the posterior, followed by MCMC in the standard parameterization to refine it. The {py:meth}`~harv.samplers.Samples.thiele_innes_to_campbell` conversion is the bridge between them.

With only a few dozen observations the period posterior is often multi-modal; see the {doc}`getting started tutorial <0-getting-started>` for a discussion of interpreting multi-modal astrometry posteriors. For data with more observations, higher signal-to-noise, or a more sophisticated noise model (jitter, instrumental offsets, Gaussian-process trends), the later tutorials show how to extend the model.